In [4]:
import pandas as pd

In [6]:
df=pd.read_csv("final_filtered_data.csv")
df.sample(5)

,id,product_search_description,meta_data
29864,54800,men apparel bottomwear jeans blue summer casua...,"{'id': 54800, 'name': 'Peter England Men Blue ..."
2643,29140,women accessories eyewear sunglasses purple wi...,"{'id': 29140, 'name': 'Vogue Women Purple Sung..."
16683,5358,men footwear shoes sports shoes white summer s...,"{'id': 5358, 'name': 'Kipsta Calcetto 5 Sr Whi..."
27704,55004,women personal care lips lipstick multi spring...,"{'id': 55004, 'name': 'Lakme 6 in 1 Travel Kit..."
9695,53444,men footwear flip flops flip flops black summe...,"{'id': 53444, 'name': 'ADIDAS Men Black Duramo..."


In [10]:
import faiss
from pathlib import Path

from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# embedding model
embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# dimension
embedding_dim = len(embedding_model.embed_query("hello world"))

# FAISS index
index = faiss.IndexFlatL2(embedding_dim)

# docstore
docstore = InMemoryDocstore()

# mapping
index_to_docstore_id = {}

# create vector store
vector_store = FAISS(
    embedding_function=embedding_model,
    index=index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id,
)

# example documents
docs = [
    Document(page_content="Delhi is the capital of India"),
    Document(page_content="Python is used for AI"),
]

# add docs
vector_store.add_documents(docs)

# save directory
save_dir = Path("vector_data/faiss")
save_dir.mkdir(parents=True, exist_ok=True)

# save FAISS
vector_store.save_local(str(save_dir))

print("FAISS index saved successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4989.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS index saved successfully!


In [11]:



vector_store = FAISS.load_local(
    "vector_data/faiss",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS loaded!")

FAISS loaded!


In [12]:
vector_store

In [13]:
retreiver=vector_store.as_retriever()

In [16]:
retreiver.invoke("Hello ",k=3)

[Document(id='59c64ee8-a81d-422a-8f73-77df4e13ce7f', metadata={}, page_content='Python is used for AI'),
 Document(id='ff18ce3a-2257-4ac8-8b6c-92d239b3269b', metadata={}, page_content='Delhi is the capital of India')]